In [4]:
import io
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def read_tga(fn):
    text = (RAW / fn).read_text(encoding="utf-8-sig", errors="replace")
    pos = text.index('\n"ARTG ID"') + 1
    return pd.read_csv(io.StringIO(text[pos:]), dtype=str, index_col=False)

tga_a = read_tga("tga_shortages_active_20261106.csv")
tga_r = read_tga("tga_shortages_archived_20261106.csv")

for name, df in [("ACTIVE", tga_a), ("ARCHIVED", tga_r)]:
    print("=====", name, df.shape)
    print(df.isna().sum().to_string())
    print("unique ARTG ID:", df["ARTG ID"].nunique(), "| unique Active ingredients:", df["Active ingredients"].nunique())
    for c in ["Shortage status", "Shortage impact rating", "Availability", "Reason"]:
        if c in df.columns:
            print(df[c].value_counts(dropna=False).to_string())
    for c in ["Supply impact start date", "Supply impact end date", "Deletion from market"]:
        d = pd.to_datetime(df[c], format="%d/%m/%Y", errors="coerce")
        print(c, "| min:", d.min(), "| max:", d.max(), "| null:", d.isna().sum(), "| unparsed:", (d.isna() & df[c].notna()).sum())

print(tga_r[tga_r["ARTG ID"].isna() | tga_r["ARTG name"].isna()].to_string())

===== ACTIVE (1044, 16)
ARTG ID                             0
ARTG name                           0
Active ingredients                  0
Dosage form                         0
Quantity of active ingredients      0
Sponsor                             0
Phone                               0
Shortage status                     0
Supply impact start date          292
Supply impact end date            292
Deletion from market              751
Shortage impact rating              0
Availability                        0
Reason                              0
Management action                 221
Last updated                        0
unique ARTG ID: 1044 | unique Active ingredients: 466
Shortage status
Current         448
Discontinued    292
Resolved        213
Anticipated      91
Shortage impact rating
Low         769
Medium      178
Critical     97
Availability
Unavailable                                      319
Available                                        304
Reduction in supply until su

In [5]:
dos_files = {"2024-25": "pbs_dos_2024-07_2025-06.csv", "2025-26": "pbs_dos_2025-07_2026-06.csv"}
dos = {k: pd.read_csv(RAW / v, dtype={"MONTH_OF_SUPPLY": str, "ITEM_CODE": str}) for k, v in dos_files.items()}
imap = pd.read_csv(RAW / "pbs_item_drug_map.csv", dtype=str, encoding="latin-1")

for k, df in dos.items():
    print("=====", k, df.shape)
    print(df.dtypes.to_string())
    print(df.isna().sum().to_string())
    print("months:", df["MONTH_OF_SUPPLY"].min(), "->", df["MONTH_OF_SUPPLY"].max())
    for c in ["PHRMCY_TYPE", "PATIENT_CAT", "DRUG_TYPE", "SCRIPT_TYPE"]:
        print(df[c].value_counts(dropna=False).to_string())
    print(df["PRESCRIPTIONS"].describe().to_string())

print("===== ITEM MAP", imap.shape)
print(imap.isna().sum().to_string())
print("unique ITEM_CODE:", imap["ITEM_CODE"].nunique(), "| unique DRUG_NAME:", imap["DRUG_NAME"].nunique())
print(imap["DRUG_NAME"].value_counts().head(10).to_string())
all_items = pd.concat(dos.values())["ITEM_CODE"].unique()
print("DOS ITEM_CODE:", len(all_items), "| found in map:", pd.Series(all_items).isin(imap["ITEM_CODE"]).mean())

===== 2024-25 (389050, 12)
MONTH_OF_SUPPLY            str
ITEM_CODE                  str
DRUG_TYPE                  str
PATIENT_CAT                str
PHRMCY_TYPE                str
SCRIPT_TYPE                str
PRESCRIPTIONS            int64
PATIENT_CONTRIB        float64
GOVT_CONTRIB           float64
TOTAL_COST             float64
RETAIL_MARKUP          float64
PATIENT_NET_CONTRIB    float64
MONTH_OF_SUPPLY        0
ITEM_CODE              0
DRUG_TYPE              0
PATIENT_CAT            0
PHRMCY_TYPE            0
SCRIPT_TYPE            0
PRESCRIPTIONS          0
PATIENT_CONTRIB        0
GOVT_CONTRIB           0
TOTAL_COST             0
RETAIL_MARKUP          0
PATIENT_NET_CONTRIB    0
months: 202407 -> 202506
PHRMCY_TYPE
S90        234374
S94 PUB     79530
S94 PRI     65845
S92          9301
PATIENT_CAT
G2    118457
C1     99660
C0     64780
R1     45702
R0     35109
G1     24866
DB       476
DRUG_TYPE
GE    362061
PL     11509
R1      6175
DT      3969
OT      3138
EP      1568
D

In [6]:
tga_names = set(pd.concat([tga_a, tga_r])["Active ingredients"].dropna().str.upper().str.strip())
pbs_names = set(imap["DRUG_NAME"].dropna().str.upper().str.strip())
print("TGA ingredient strings:", len(tga_names), "| exact match in PBS DRUG_NAME:", len(tga_names & pbs_names))
print(sorted(tga_names - pbs_names)[:15])
print(sorted(pbs_names - tga_names)[:15])

TGA ingredient strings: 1351 | exact match in PBS DRUG_NAME: 405
['ABACAVIR SULFATE', 'ABACAVIR SULFATE~LAMIVUDINE', 'ABIRATERONE ACETATE', 'ACAMPROSATE CALCIUM', 'ACETAZOLAMIDE SODIUM', 'ACETYLCHOLINE CHLORIDE', 'ACETYLCHOLINE CHLORIDE~MANNITOL', 'ACICLOVIR SODIUM DIHYDRATE', 'ACTIVATED CHARCOAL~SORBITOL', 'ADAPALENE', 'ADEFOVIR DIPIVOXIL', 'ADENOSINE', 'ADRENALINE (EPINEPHRINE) ACID TARTRATE', 'ADRENALINE (EPINEPHRINE) ACID TARTRATE~ARTICAINE HYDROCHLORIDE', 'ADRENALINE (EPINEPHRINE) ACID TARTRATE~BUPIVACAINE HYDROCHLORIDE']
['ABACAVIR', 'ABACAVIR + LAMIVUDINE', 'ABACAVIR + LAMIVUDINE + ZIDOVUDINE', 'ABALOPARATIDE', 'ABCIXIMAB', 'ABEMACICLIB', 'ABIRATERONE', 'ABIRATERONE (&) METHYLPREDNISOLONE', 'ABIRATERONE (&) PREDNISOLONE', 'ABSORBENT PAD PANTS', 'ABSORBENT PADS', 'ACAMPROSATE', 'ACETIC ACID', 'ACETIC ACID + OXYQUINOLINE + RICINOLEIC ACID', 'ACLIDINIUM']
